In [55]:
YEAR = 2020

from enum import Enum
class DisplayType(Enum):
    TOTAL_TAX = "税"
    TOTAL_REMAINING = "剩余收入"
    REMAINING_PERCENTAGE = "剩余收入百分比"

    def resultFor(self, salaryBonus, equity) -> float:
        healthInsurance = salaryBonus * 0.04
        welfare = salaryBonus * 0.08
        if self == DisplayType.TOTAL_TAX:
            nationalTax = nationalTaxToBePaid2020(preTaxSalaryIncome=salaryBonus * 10000,
                                                equityIncome=equity * 10000,
                                                withholding=0,
                                                socialSecurity=welfare * 10000,
                                                otherDeduction=healthInsurance * 10000)
            localTax = shibuyaLocalTax2020(preTaxSalaryIncome=salaryBonus * 10000,
                                        equityIncome=equity * 10000,
                                        socialSecurity=welfare * 10000,
                                        otherDeduction=healthInsurance * 10000)
            totalTax = nationalTax + localTax
            # Divide twice to round down one digit after decimal
            return int(totalTax / 1000) / 10
        if self == DisplayType.TOTAL_REMAINING:
            totalTax = DisplayType.TOTAL_TAX.resultFor(salaryBonus, equity)
            return salaryBonus + equity - totalTax - healthInsurance - welfare
        if self == DisplayType.REMAINING_PERCENTAGE:
            remaining = DisplayType.TOTAL_REMAINING.resultFor(salaryBonus, equity)
            perc = (remaining / (salaryBonus + equity)) * 100
            return perc
        assert(False)

salaryBonusRange = range(700, 1801, 50)
equityRange = range(50, 801, 50)

zValues = []
for _ in range(len(salaryBonusRange)):
    zValues.append([0] * len(equityRange))

# displayType = DisplayType.TOTAL_TAX
# displayType = DisplayType.TOTAL_REMAINING
displayType = DisplayType.REMAINING_PERCENTAGE

from itertools import product
from taxCalculation import nationalTaxToBePaid2020, shibuyaLocalTax2020
for salaryBonusWithIdx, equityWithIdx in product(enumerate(salaryBonusRange), enumerate(equityRange)):
    salaryBonusIdx, salaryBonus = salaryBonusWithIdx
    equityIdx, equity = equityWithIdx
    zValues[salaryBonusIdx][equityIdx] = displayType.resultFor(salaryBonus, equity)

import plotly.graph_objects as go
fig = go.Figure(data=[go.Surface(z=zValues, y=list(salaryBonusRange), x=list(equityRange))])
scene = dict(yaxis_title='薪水和奖金',
             xaxis_title='股权',
             zaxis_title=displayType.value,
             zaxis=dict(rangemode="tozero"))
fig.update_layout(height=600, scene=scene)
fig.show()

In [60]:
equity = 300
salaryBonusRange = range(800, 1801, 10)
totals = [sb + equity for sb in salaryBonusRange]
results = [displayType.resultFor(sb, equity) for sb in salaryBonusRange]

fig = go.Figure()
# fig.add_trace(go.Scatter(x=list(salaryBonusRange), y=totals, mode="lines", name="税前收入"))
fig.add_trace(go.Scatter(x=list(salaryBonusRange), y=results, mode="lines", name=displayType.value))
fig.update_layout(yaxis=dict(rangemode="tozero"), xaxis_title="工资和奖金", showlegend=True)